Création de la base de données et les schémas

In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE DATABASE IF NOT EXISTS HOUSE_PRICE;
CREATE SCHEMA IF NOT EXISTS HOUSE_PRICE.BRONZE;
CREATE SCHEMA IF NOT EXISTS HOUSE_PRICE.SILVER;

USE SCHEMA HOUSE_PRICE.BRONZE;


Création du stage

In [ ]:
CREATE OR REPLACE STAGE house_price_stage
URL = 's3://logbrain-datalake/datasets/house_price/'
FILE_FORMAT = (TYPE = JSON);

LIST @house_price_stage;

Création de la table raw

In [ ]:
CREATE OR REPLACE TABLE house_price_raw (
    data VARIANT
);

Charger les données (COPY INTO sans FILE FORMAT)

In [ ]:
COPY INTO house_price_raw
FROM @house_price_stage
FILE_FORMAT = (
    TYPE = JSON,
    STRIP_OUTER_ARRAY = TRUE
)
ON_ERROR = CONTINUE;

select count(*) as total from house_price_raw;

Créer la table structurée

In [ ]:
CREATE OR REPLACE TABLE house_price (
    price NUMBER DEFAULT 0,                         -- Prix de vente
    area NUMBER DEFAULT 0,                          -- Surface en m²
    bedrooms NUMBER DEFAULT 0,                      -- Chambres
    bathrooms NUMBER DEFAULT 0,                     -- Salles de bain
    stories NUMBER DEFAULT 0,                       -- Étages
    mainroad STRING DEFAULT 'no',                   -- Reliée à route principale (yes/no)
    guestroom STRING DEFAULT 'no',                  -- Chambre d'amis (yes/no)
    basement STRING DEFAULT 'no',                   -- Sous-sol (yes/no)
    hotwaterheating STRING DEFAULT 'no',            -- Chauffage eau chaude (yes/no)
    airconditioning STRING DEFAULT 'no',            -- Climatisation (yes/no)
    parking NUMBER DEFAULT 0,                       -- Places de parking
    prefarea STRING DEFAULT 'no',                   -- Zone privilégiée (yes/no)
    furnishingstatus STRING DEFAULT 'unfurnished'   -- Meublée / semi-meublée / non meublée
);


Chargement dans house_price

In [ ]:
INSERT INTO house_price
SELECT
    COALESCE(TRY_TO_NUMBER(data:price::STRING), 0),
    COALESCE(TRY_TO_NUMBER(data:area::STRING), 0),
    COALESCE(TRY_TO_NUMBER(data:bedrooms::STRING), 0),
    COALESCE(TRY_TO_NUMBER(data:bathrooms::STRING), 0),
    COALESCE(TRY_TO_NUMBER(data:stories::STRING), 0),
    COALESCE(data:mainroad::STRING, 'no'),
    COALESCE(data:guestroom::STRING, 'no'),
    COALESCE(data:basement::STRING, 'no'),
    COALESCE(data:hotwaterheating::STRING, 'no'),
    COALESCE(data:airconditioning::STRING, 'no'),
    COALESCE(TRY_TO_NUMBER(data:parking::STRING), 0),
    COALESCE(data:prefarea::STRING, 'no'),
    COALESCE(data:furnishingstatus::STRING, 'unfurnished')
FROM house_price_raw;

select * from house_price limit 10;


Voire le contenu des données et le type

In [ ]:
SELECT data, typeof(data)
FROM house_price_raw
LIMIT 5;


se connecter à silver

In [ ]:
use SCHEMA silver;

CREATE OR REPLACE TABLE silver.house_price (
    area NUMBER,
    bedrooms NUMBER,
    bathrooms NUMBER,
    stories NUMBER,
    mainroad BOOLEAN,
    guestroom BOOLEAN,
    basement BOOLEAN,
    hotwaterheating BOOLEAN,
    airconditioning BOOLEAN,
    parking NUMBER,
    prefarea BOOLEAN,
    furnishingstatus STRING,
    price NUMBER,
    price_per_m2 NUMBER
);


 Nettoyage, transformation et insertion dans Silver

In [ ]:
INSERT INTO silver.house_price (
    area,
    bedrooms,
    bathrooms,
    stories,
    mainroad,
    guestroom,
    basement,
    hotwaterheating,
    airconditioning,
    parking,
    prefarea,
    furnishingstatus,
    price,
    price_per_m2
)
SELECT
    TRY_TO_NUMBER(data:area::STRING) AS area,
    TRY_TO_NUMBER(data:bedrooms::STRING) AS bedrooms,
    TRY_TO_NUMBER(data:bathrooms::STRING) AS bathrooms,
    TRY_TO_NUMBER(data:stories::STRING) AS stories,

    IFF(LOWER(data:mainroad::STRING) = 'yes', TRUE, FALSE) AS mainroad,
    IFF(LOWER(data:guestroom::STRING) = 'yes', TRUE, FALSE) AS guestroom,
    IFF(LOWER(data:basement::STRING) = 'yes', TRUE, FALSE) AS basement,
    IFF(LOWER(data:hotwaterheating::STRING) = 'yes', TRUE, FALSE) AS hotwaterheating,
    IFF(LOWER(data:airconditioning::STRING) = 'yes', TRUE, FALSE) AS airconditioning,

    TRY_TO_NUMBER(data:parking::STRING) AS parking,
    IFF(LOWER(data:prefarea::STRING) = 'yes', TRUE, FALSE) AS prefarea,

    LOWER(data:furnishingstatus::STRING) AS furnishingstatus,

    TRY_TO_NUMBER(data:price::STRING) AS price,

    TRY_TO_NUMBER(data:price::STRING) / NULLIF(TRY_TO_NUMBER(data:area::STRING), 0) AS price_per_m2
FROM bronze.house_price_raw;


Vérification 

In [ ]:
SELECT * FROM silver.house_price LIMIT 20;

Initialisation de Snowpark

In [ ]:
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, avg, count, corr, stddev, min, max

session = Session.builder.getOrCreate()

Charger la table Bronze dans un DataFrame Snowpark

In [ ]:
df_raw = session.table("bronze.house_price_raw")
df_raw.show()

Aperçu général du dataset

In [ ]:
df_raw.count()

Schéma

In [ ]:
df_raw.schema
df_raw.show(10)

Statistiques descriptives

In [ ]:
df_raw.describe().show()

Analyse des corrélations

In [ ]:
df_corr = df_raw.select(
    corr(col("DATA")["price"].cast("float"), col("DATA")["area"].cast("float")).alias("corr_price_area"),
    corr(col("DATA")["price"].cast("float"), col("DATA")["bedrooms"].cast("float")).alias("corr_price_bedrooms"),
    corr(col("DATA")["price"].cast("float"), col("DATA")["bathrooms"].cast("float")).alias("corr_price_bathrooms")
)

df_corr.show()


In [ ]:
df_clean = df_raw.select(
    col("DATA")["area"].cast("float").alias("area"),
    col("DATA")["bedrooms"].cast("int").alias("bedrooms"),
    col("DATA")["bathrooms"].cast("int").alias("bathrooms"),
    col("DATA")["stories"].cast("int").alias("stories"),
    (col("DATA")["mainroad"] == "yes").alias("mainroad"),
    (col("DATA")["guestroom"] == "yes").alias("guestroom"),
    (col("DATA")["basement"] == "yes").alias("basement"),
    (col("DATA")["hotwaterheating"] == "yes").alias("hotwaterheating"),
    (col("DATA")["airconditioning"] == "yes").alias("airconditioning"),
    col("DATA")["parking"].cast("int").alias("parking"),
    (col("DATA")["prefarea"] == "yes").alias("prefarea"),
    col("DATA")["furnishingstatus"].alias("furnishingstatus"),
    col("DATA")["price"].cast("float").alias("price")
)

Distribution des variables

In [ ]:
import matplotlib.pyplot as plt

pdf = df_corr.to_pandas()

plt.hist(pdf["price"], bins=30)
plt.title("Distribution du prix")
plt.show()
